<h3 style="text-align: center;"><b>Школа глубокого обучения ФПМИ МФТИ</b></h3>

<h3 style="text-align: center;"><b>Домашнее задание. Детекция объектов</b></h3>

В этом домашнем задании мы продолжим работу над детектором из семинара, поэтому при необходимости можете заимствовать оттуда любой код.

Домашнее задание можно разделить на следующие части:

* Переделываем модель [4]
  * Backbone[1],
  * Neck [2],
  * Head [1]
* Label assignment [3]:
  * TAL [3]
* Лоссы [1]:
  * CIoU loss [1]
* Кто больше? [5]
  * 0.05 mAP [1]
  * 0.1 mAP  [2]
  * 0.2 mAP [5]

**Максимальный балл:** 10 баллов. (+3 балла бонус).

In [1]:
import torch
import numpy as np
import pandas as pd
import albumentations as A
import io

from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset
from albumentations.pytorch.transforms import ToTensorV2

try:
    import torchmetrics
except ImportError:
    !pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 21.4 MB/s eta 0:00:00


### Загрузка данных

Мы продолжаем работу с датасетом из семинара - Halo infinite ([сслыка](https://universe.roboflow.com/graham-doerksen/halo-infinite-angel-aim)). Загрузка данных и создание датасета полностью скопированы из семинара.

Сначала загружаем данные

In [2]:
splits = {'train': 'data/train-00000-of-00001-0d6632d599c29801.parquet',
          'validation': 'data/validation-00000-of-00001-c6b77a557eeedd52.parquet',
          'test': 'data/test-00000-of-00001-866d29d8989ea915.parquet'}
df_train = pd.read_parquet("hf://datasets/Francesco/halo-infinite-angel-videogame/" + splits["train"])
df_test = pd.read_parquet("hf://datasets/Francesco/halo-infinite-angel-videogame/" + splits["test"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Создаем датасет для предобработки данных

In [3]:
class HaloDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        df_objects = pd.json_normalize(dataframe['objects'])[["bbox", "category"]]
        df_images = pd.json_normalize(dataframe['image'])[["bytes"]]
        self.data = dataframe[["image_id"]].join(df_objects).join(df_images)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        """Загружаем данные и разметку для объекта с индексом `idx`.

        labels: List[int] Набор классов для каждого ббокса,
        boxes: List[List[int]] Набор ббоксов в формате (x_min, y_min, w, h).
        """
        row = self.data.iloc[idx]
        image = Image.open(io.BytesIO(row["bytes"]))
        image = np.array(image)

        target = {}
        target["image_id"] = row["image_id"]

        labels = [row["category"]] if isinstance(row["category"], int) else row['category']
        # Вычитаем единицу чтобы классы начинались с нуля
        labels = [label - 1 for label in labels]
        boxes = row['bbox'].tolist()

        if self.transform is not None:
            transformed = self.transform(image=image, bboxes=boxes, labels=labels)
            image, boxes, labels = transformed["image"], transformed["bboxes"], transformed["labels"]
        else:
            image = transforms.ToTensor()(image)

        target['boxes'] = torch.tensor(np.array(boxes), dtype=torch.float32)
        target['labels'] = torch.tensor(labels, dtype=torch.int64)
        return image, target

def collate_fn(batch):
    batch = tuple(zip(*batch))
    images = torch.stack(batch[0])
    return images, batch[1]

Чтобы модель не переобучалась, можно добавить больше аугментаций, весь список можно посмотреть тут [[ссылка](https://explore.albumentations.ai/)].

Какие можно использовать аугментации?
* Добавить зум `RandomResizedCrop`,
* Сделать цветовые аугментации типа `RandomBrightnessContrast` и/или `HueSaturationValue`,
* Добавить шум `GaussNoise`,
* Вырезать случайные части изображения `CoarseDropout`,
* И любые другие!

Аугментации можно комбинировать посредствам `A.OneOf`, `A.SomeOf` или `A.RandomOrder`.

Хоть аугментации ограничиваются только вашей фантазией, перед обучением советуем посмотреть на результат преобразований и убедиться, что изображение ещё поддается детекции:)

In [4]:
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)

train_transform = A.Compose(
    [
        # Добавляй сюда свои аугментации при необходимости!
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ],
    # Раскомментируй, если аугментации изменяют ббоксы.
    # Не забудь указать верный формат для ббоксов.
    # bbox_params=A.BboxParams(format='coco', label_fields=['labels'])
)

test_transform = A.Compose(
    [
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ]
)

Не забываем инициализировать наш датасет

In [5]:
train_dataset = HaloDataset(df_train, transform=train_transform)
test_dataset = HaloDataset(df_test, transform=test_transform)

## Переделываем модель [4 балла]

В семинаре мы реализовали самый базовый детектор, а сейчас настало время его улучшать.

### Backbone [1 балл]

Хорошей практикой считается размораживать несколько последних слоев в backbone, это позволяет немного улучить качество модели. Давайте улушчим класс Backbone из лекции, добавив ему возможность разморозки __k__ последних слоев или блоков (на ваш выбор).

In [6]:
import torch.nn as nn
from torchvision import models

class Backbone(nn.Module):
    def __init__(self, model_name='resnet50', unfreeze_last=2):
        super().__init__()
        if model_name == 'resnet50':
            base_model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        else:
            raise ValueError("Unsupported model name")

        modules = list(base_model.children())[:-2]
        self.backbone = nn.Sequential(*modules)

        for param in self.backbone.parameters():
            param.requires_grad = False

        if unfreeze_last > 0:
            for child in list(self.backbone.children())[-unfreeze_last:]:
                for param in child.parameters():
                    param.requires_grad = True

    def forward(self, x):
        return self.backbone(x)

### NECK [2 балла]

Следующее улучшение коснется шеи. Предлагаем реализовать знакомую из лекции архитектуру FPN.

#### Feature Pyramid Network

<center><img src="https://user-images.githubusercontent.com/57972646/69858594-b14a6c00-12d5-11ea-8c3e-3c17063110d3.png"/></center>


* [Feature Pyramid Networks for Object Detection](https://arxiv.org/abs/1612.03144)

Она состоит из top-down пути, в котором происходит 2 вещи:
1. Увеличивается пространственная размерность фичей,
2. С помощью скипконнекшеннов, добавляются фичи из backbone модели.

Для увеличения пространственной размерности используется __nearest neighbor upsampling__, а фичи из шеи и бекбоуна суммируются.

__TIPS__:
* Можете использовать базовые классы из лекции,
* Воспользуйтесь AnchorGenerator-ом, чтобы создавать якоря сразу для нескольких выходов,
* Не забудьте использовать nn.ModuleList, если захотите сделать динамическое количество голов у модели,
* Также, можно добавить доп конволюцию (3х3 с паддингом) у каждого выхода шеи.

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Neck(nn.Module):
    def __init__(self, in_channels_list=[512, 1024, 2048], out_channels=256):
        super().__init__()
        self.lateral_convs = nn.ModuleList([
            nn.Conv2d(in_channels, out_channels, kernel_size=1)
            for in_channels in in_channels_list
        ])

        self.fpn_convs = nn.ModuleList([
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
            for _ in range(len(in_channels_list))
        ])

        self.p6_conv = nn.Conv2d(in_channels_list[-1], out_channels, kernel_size=3, stride=2, padding=1)

    def forward(self, features):
        laterals = [conv(f) for conv, f in zip(self.lateral_convs, features)]

        for i in range(len(laterals) - 1, 0, -1):
            upsampled = F.interpolate(laterals[i], size=laterals[i-1].shape[-2:], mode='nearest')
            laterals[i-1] = laterals[i-1] + upsampled

        outs = [self.fpn_convs[i](laterals[i]) for i in range(len(laterals))]

        p6 = self.p6_conv(features[-1])
        outs.append(p6)

        return outs

### Head [1 балл]

В качестве шеи можно выбрать __один из двух__ вариантов:

#### 1. Decoupled Head

Реализовать Decoupled Head из [YOLOX](https://arxiv.org/abs/2107.08430).
<center><img src="https://i.ibb.co/BVtBR2R3/Decoupled-head.jpg"/></center>

**TIP**: Возьмите за основу голову из семинара, тк она сильно похожа на Decoupled Head.

Изменять количество параметров у шей на разных уровнях не обязательно.

#### 2. Confidence score free head

Нужно взять за основу голову из семинара и полностью убрать предсказание confidence score. Чтобы модель предсказывала только 2 группы: ббоксы и классы.

Есть следующие способы удаления confidence score:
* Добавление нового класса ФОН. Обычно его обозначают нулевым классом.
* Присваивание ббоксам БЕЗ объекта вектор из нулей в качестве таргета.

Выберете тот, который вам больше нравится и будте внимательны при расчете лосса!

**Важно!** Удаление confidence score повлияет на следующие методы из семинара:
* target_assign
* ComputeLoss
* _filter_predictions

In [8]:
import torch
import torch.nn as nn

class Head(nn.Module):
    def __init__(self, in_channels=256, num_classes=1, num_anchors=1):
        super().__init__()
        self.num_classes = num_classes
        self.num_anchors = num_anchors

        self.stems = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

        self.cls_convs = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

        self.reg_convs = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

        self.cls_preds = nn.Conv2d(in_channels, num_anchors * num_classes, kernel_size=1)
        self.reg_preds = nn.Conv2d(in_channels, num_anchors * 4, kernel_size=1)
        self.obj_preds = nn.Conv2d(in_channels, num_anchors * 1, kernel_size=1)

    def forward(self, x):
        x = self.stems(x)

        cls_feat = self.cls_convs(x)
        reg_feat = self.reg_convs(x)

        cls_output = self.cls_preds(cls_feat)
        reg_output = self.reg_preds(reg_feat)
        obj_output = self.obj_preds(reg_feat)

        return cls_output, reg_output, obj_output

Теперь можно снова реализовать класс детектора с учетом всех частей выше!

In [9]:
class Detector(nn.Module):
    def __init__(self, num_classes=1, unfreeze_last=2):
        super().__init__()
        base_resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

        self.backbone_p1 = nn.Sequential(*list(base_resnet.children())[:5])
        self.backbone_p2 = list(base_resnet.children())[5]
        self.backbone_p3 = list(base_resnet.children())[6]
        self.backbone_p4 = list(base_resnet.children())[7]

        for layer in [self.backbone_p1, self.backbone_p2, self.backbone_p3, self.backbone_p4]:
            for param in layer.parameters():
                param.requires_grad = False

        if unfreeze_last > 0:
            for param in self.backbone_p4.parameters():
                param.requires_grad = True

        self.neck = Neck(in_channels_list=[512, 1024, 2048], out_channels=256)

        self.head = Head(in_channels=256, num_classes=num_classes, num_anchors=1)

    def forward(self, x):
        x = self.backbone_p1(x)
        c3 = self.backbone_p2(x)
        c4 = self.backbone_p3(c3)
        c5 = self.backbone_p4(c4)

        fpn_features = self.neck([c3, c4, c5])

        outputs = []
        for feat in fpn_features:
            cls_out, reg_out, obj_out = self.head(feat)
            outputs.append({
                'cls': cls_out,
                'reg': reg_out,
                'obj': obj_out
            })

        return outputs

## Label assignment [3 балла]
В этой секции предлагается заменить функцию `assign_target` на более современный алгоритм который называется Task alignment learning.

Он описан в статье [TOOD](https://arxiv.org/abs/2108.07755) в секции 3.2. Для удобства вот его основные шаги:

1. Посчитать значение метрики для каждого предсказанного ббокса:
    
$$t = s^\alpha * u^\beta$$
    
где,
* $s$ — classification score, или вероятность принадлежности предсказанного ббокса к классу реального ббокса (**GT**);
* $u$ — IoU между предсказанным и реальным ббоксами;
* $\alpha,\ \beta$ — нормализационные константы, обычно $\alpha = 6.0, \ \beta = 1.0$.
    
2. Отфильтровать предсказания на основе **GT**.

    Для якорных детекторов, обычно, выбираются только те предсказания, центры якорей которых находятся внутри GT.
4. Для каждого **GT** выбрать несколько (обычно 5 или 13) самых подходящих предсказаний.
5. Если предсказание рассматривается в качестве подходящего для нескольких **GT** — выбрать **GT** с наибольшим пересечением по IoU.


**BAЖНО**: если будете использовать Runner из лекции, не забудьте поменять параметры  в `self.assign_target_method` в методе `_run_train_epoch`.

In [10]:
import torch
import torch.nn.functional as F
from torchvision.ops import box_iou

def TAL_assigner(pred_scores, pred_bboxes, gt_labels, gt_bboxes, alpha=6.0, beta=1.0, topk=13):
    """
    Task Alignment Learning Assigner.
    pred_scores: [num_anchors, num_classes]
    pred_bboxes: [num_anchors, 4]
    gt_labels: [num_gt]
    gt_bboxes: [num_gt, 4]
    """
    num_anchors = pred_bboxes.size(0)
    num_gt = gt_bboxes.size(0)

    if num_gt == 0:
        return torch.zeros(num_anchors, dtype=torch.long), torch.zeros((num_anchors, 4))

    # Расчет IoU
    ious = box_iou(gt_bboxes, pred_bboxes) # [num_gt, num_anchors]

    # Выбираем скоры для соответствующих GT классов
    # Если у нас 1 класс, gt_labels всегда будут 0
    gt_labels_indices = gt_labels.long().clamp(max=pred_scores.size(1) - 1)
    pred_scores_selected = pred_scores[:, gt_labels_indices].t() # [num_gt, num_anchors]

    # Вычисляем метрику выравнивания
    alignment_metrics = (pred_scores_selected ** alpha) * (ious ** beta)

    # Top-k стратегия
    actual_topk = min(topk, num_anchors)
    topk_metrics, topk_indices = torch.topk(alignment_metrics, actual_topk, dim=1)
    is_in_topk = torch.zeros_like(alignment_metrics, dtype=torch.bool)
    is_in_topk.scatter_(1, topk_indices, True)

    alignment_metrics = alignment_metrics * is_in_topk.float()

    # Если один якорь подходит нескольким GT, выбираем лучший
    max_vals, target_gt_idx = alignment_metrics.max(dim=0)
    mask_best_gt = max_vals > 0

    assigned_labels = torch.zeros(num_anchors, dtype=torch.long)
    # Сохраняем метку + 1 (0 зарезервирован под фон)
    assigned_labels[mask_best_gt] = gt_labels[target_gt_idx[mask_best_gt]] + 1

    assigned_bboxes = torch.zeros((num_anchors, 4))
    assigned_bboxes[mask_best_gt] = gt_bboxes[target_gt_idx[mask_best_gt]]

    return assigned_labels, assigned_bboxes

### DIoU [1]

Вместо SmoothL1, который используется в семинаре, реализуем лосс, основанный на пересечении ббоксов. В качестве тренировки давайте напишем Distance Intersection over Union (DIoU).

<center><img src=https://wikidocs.net/images/page/163613/Free_Fig_5.png></center>

Для его реализации разобъем задачу на части:

**1. Реализуем IoU:**

Пусть даны координаты для предсказанного ($B^p$) и истинного ($B^g$) ббоксов в формате XYXY или VOC PASCAL (левый верхний и правый нижний углы):

$B^p=(x^p_1, y^p_1, x^p_2, y^p_2)$, $B^g=(x^g_1, y^g_1, x^g_2, y^g_2)$, тогда алгоритм расчета будет следующий:

    1. Найдем площади обоих ббоксов:
$$ A^p = (x^p_2 - x^p_1) * (y^p_2 - y^p_1) $$
$$ A^g = (x^g_2 - x^g_1) * (y^g_2 - y^g_1) $$

    2. Посчитаем пересечение между ббоксами:

Тут мы предлагаем вам подумать как в общем виде можно расчитать размеры ббокса, который будет являться пересечением $B^p$ и $B^g$, а затем посчитать его площадь:

$$x^I_1 = \qquad \qquad y^I_1 = $$
$$x^I_2 = \qquad \qquad y^I_2 = $$

В общем виде, площать будет записываться следующим образом:

Если $x^I_2 > x^I_1$ & $y^I_2 > y^I_1$, тогда:

$$I = (x^I_2 - x^I_1) * (y^I_2 - y^I_1)$$

Иначе, $I = 0$.

    3. Считаем объединение ббоксов.

Мы можем посчитать эту площадь как сумму площадей двух ббоксов минус площадь пересечения (тк мы считаем её два раз в сумме площадей):

$$U = A^p + A^g - I$$

    4. Вычисляем IoU.

$$IoU = \frac{I}{U}$$

**2. Посчитаем диагональ выпуклой оболочки:**

Для расчета диагонали, сначала выпишите координаты верхнего левого и правого нижнего углов. Подумайте, чему будут равны эти координаты в общем случае?

$$x^c_1 = \qquad \qquad y^c_1 = $$
$$x^c_2 = \qquad \qquad y^c_2 = $$

Подсказка: Нарисуйте несколько вариантов пересечений предсказания и GT на бумажке, и выпишите координаты для выпуклой оболочки.

Тогда квадрат диагонали можно посчитать по формуле:

$$c^2 = (x^c_2 - x^c_1)^2 + (y^c_2 - y^c_1)^2$$

**3. Рассчитаем расстояние между цетрами ббоксов:**

Сначала находим координаты центров каждого из ббоксов (если ббоксы в формате YOLO, то и считать ничего не нужно), затем считаем Евклидово расстояние между центрами.

$d = $

Собираем все части вместе и считаем лосс по формуле:

$$ DIoU = 1 - IoU + \frac{d^2}{c^2}$$

Помните, что пар ббоксов может быть много! Возвращайте усредненное значение лосса.

In [11]:
from torchvision.ops import distance_box_iou_loss

In [12]:
def gen_bbox(num_boxes=10):
    min_corner = torch.randint(0, 100, (num_boxes, 2))
    max_corner = torch.randint(50, 150, (num_boxes, 2))

    for i in range(2):
        wrong_order = min_corner[:, i] > max_corner[:, i]
        if wrong_order.any():
            min_corner[wrong_order, i], max_corner[wrong_order, i] = max_corner[wrong_order, i], min_corner[wrong_order, i]
    return torch.cat((min_corner, max_corner), dim=1)

In [13]:
pred_boxes = gen_bbox(num_boxes=100)
true_boxes = gen_bbox(num_boxes=100)

In [14]:
print(f" DIoU: {distance_box_iou_loss(pred_boxes, true_boxes, reduction="mean").item()}")

 DIoU: 0.9852734208106995


In [15]:
def diou_loss(pred_boxes, gt_boxes):
    """
    Расчет Distance IoU Loss.
    pred_boxes, gt_boxes: тензоры формата [N, 4] (xyxy)
    """
    x1_p, y1_p, x2_p, y2_p = pred_boxes.unbind(-1)
    x1_g, y1_g, x2_g, y2_g = gt_boxes.unbind(-1)

    area_p = (x2_p - x1_p) * (y2_p - y1_p)
    area_g = (x2_g - x1_g) * (y2_g - y1_g)

    x1_i = torch.max(x1_p, x1_g)
    y1_i = torch.max(y1_p, y1_g)
    x2_i = torch.min(x2_p, x2_g)
    y2_i = torch.min(y2_p, y2_g)

    intersection = (x2_i - x1_i).clamp(min=0) * (y2_i - y1_i).clamp(min=0)
    union = area_p + area_g - intersection
    iou = intersection / union.clamp(min=1e-6)

    center_x_p = (x1_p + x2_p) / 2
    center_y_p = (y1_p + y2_p) / 2
    center_x_g = (x1_g + x2_g) / 2
    center_y_g = (y1_g + y2_g) / 2

    d2 = (center_x_p - center_x_g)**2 + (center_y_p - center_y_g)**2

    x1_c = torch.min(x1_p, x1_g)
    y1_c = torch.min(y1_p, y1_g)
    x2_c = torch.max(x2_p, x2_g)
    y2_c = torch.max(y2_p, y2_g)

    c2 = (x2_c - x1_c)**2 + (y2_c - y1_c)**2

    diou = iou - d2 / c2.clamp(min=1e-6)
    loss = 1 - diou

    return loss.mean()

In [16]:
import numpy as np
pred_boxes = gen_bbox(num_boxes=1000)
true_boxes = gen_bbox(num_boxes=1000)

# проверим что написанный лосс выдает те же результаты что и лосс из торча.
assert np.isclose(diou_loss(pred_boxes, true_boxes), distance_box_iou_loss(pred_boxes, true_boxes, reduction="mean"))

## Кто больше? [5 баллов]

Наконец то мы дошли до самый интересной части. Тут мы раздаем очки за mAP'ы!

Все что вы написали выше вам поможет улучшить качество итогового детектора, настало время узнать насколько сильно :)

За достижения порога по mAP на тестовом наборе вы получаете баллы:
* 0.05 mAP [1]
* 0.1 mAP [2]
* 0.2 mAP [5]


**TIPS**:
1. На семинаре мы специально не унифицировали формат ббоксов между методами, чтобы обратить ваше внимание что за этим нужно следить. Чтобы было проще, сразу унифицируете формат по всему ноутбуку. Советуем использовать формат xyxy, тк IoU и NMS из torch используют именно этот формат. (Не забудьте поменять формат у таргета в `HaloDataset`).

2. Попробуйте перейти к IoU-based лоссу при обучении. То есть обучать не смещения, а сразу предсказывать ббокс.

3. Поэксперементируйте с подходами target assignment'а в процессе обучения. Например, можно на первых итерациях использовать обычный метод, а затем подключить TAL.

4. Добавьте аугментаций!

Можно взять [albumentations](https://albumentations.ai/docs/getting_started/bounding_boxes_augmentation/), библиотеку, которую мы использовали всеминаре. Или базовые аугментации из торча [тык](https://pytorch.org/vision/main/transforms.html). Если будете использовать торч, не забудте про ббоксы, transforms из коробки не будет их агументировать.

5. Можете реализовать другую шею, которую мы обсуждали на лекции [Path Aggregation Network](https://arxiv.org/abs/1803.01534) она точно улучшит ваше итоговое качество.

6. Попробуйте добавлять различные блоки из YOLO архитектур в шею вместо единичных конволюционных слоев. (Например, замените конволюции 3х3 на CSP блоки).

7. Попробуйте заменить NMS на другой метод (WeightedNMS, SoftNMS, etc.). Немного ссылок:
    * Статья про SoftNMS [тык](https://arxiv.org/pdf/1704.04503)
    * Статья про WeightedNMS [тык](https://openaccess.thecvf.com/content_ICCV_2017_workshops/papers/w14/Zhou_CAD_Scale_Invariant_ICCV_2017_paper.pdf)
    * Есть их реализация, правда на нумбе [git](https://github.com/ZFTurbo/Weighted-Boxes-Fusion?tab=readme-ov-file)

8. Не бойтесь эксперементировать и удачи!

Также, напишите развернутые ответы на следующие вопросы:

**Questions:**
1. Какой метод label assignment'a помогает лучше обучаться модели? Почему?
2. Какое из сделаных вами улучшений внесло наибольший вклад в качество модели? Как вы думаете, почему это произошло?
3. Какое из сделанных вами улучшений вообще не изменило метрику? Как вы думаете, почему это произошло?

In [17]:
import torch
import torch.nn as nn

class AnchorGenerator:
    def __init__(self, strides=[8, 16, 32, 64]):
        self.strides = strides

    def __call__(self, features):
        anchors = []
        for feat, stride in zip(features, self.strides):
            h, w = feat.shape[-2:]
            shift_x = torch.arange(0, w) * stride
            shift_y = torch.arange(0, h) * stride
            shift_y, shift_x = torch.meshgrid(shift_y, shift_x, indexing='ij')

            anchor_points = torch.stack([shift_x, shift_y], dim=-1).to(feat.device)
            anchor_points = anchor_points.reshape(-1, 2) + stride // 2
            anchors.append(anchor_points)
        return torch.cat(anchors, dim=0)

class ComputeLoss(nn.Module):
    def __init__(self, num_classes=1):
        super().__init__()
        self.num_classes = num_classes
        self.bce_cls = nn.BCEWithLogitsLoss(reduction='sum')

    def forward(self, predictions, targets, anchors_points):
        all_cls = torch.cat([p['cls'].flatten(2).transpose(1, 2) for p in predictions], dim=1)
        all_reg = torch.cat([p['reg'].flatten(2).transpose(1, 2) for p in predictions], dim=1)
        all_obj = torch.cat([p['obj'].flatten(2).transpose(1, 2) for p in predictions], dim=1)

        batch_size = all_cls.size(0)
        loss_cls, loss_reg, loss_obj = 0, 0, 0

        for i in range(batch_size):
            pred_scores = torch.sigmoid(all_cls[i]) * torch.sigmoid(all_obj[i])
            pred_bboxes = all_reg[i]

            gt_boxes = targets[i]['boxes'].to(all_cls.device)
            gt_labels = targets[i]['labels'].to(all_cls.device)

            if len(gt_boxes) == 0:
                loss_obj += nn.BCEWithLogitsLoss()(all_obj[i], torch.zeros_like(all_obj[i]))
                continue

            target_labels, target_bboxes = TAL_assigner(
                pred_scores.detach(), pred_bboxes.detach(), gt_labels, gt_boxes
            )

            mask_pos = (target_labels > 0)

            t_cls = torch.zeros_like(all_cls[i])
            if mask_pos.any():
                idx = target_labels[mask_pos] - 1
                t_cls[mask_pos, idx] = 1.0
            loss_cls += self.bce_cls(all_cls[i], t_cls)

            if mask_pos.any():
                loss_reg += diou_loss(pred_bboxes[mask_pos], target_bboxes[mask_pos])

            t_obj = mask_pos.float().unsqueeze(-1)
            loss_obj += nn.BCEWithLogitsLoss()(all_obj[i], t_obj)

        return (loss_cls + loss_reg + loss_obj) / batch_size

In [18]:
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Detector(num_classes=1).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
criterion = ComputeLoss(num_classes=1)
anthropomorph = AnchorGenerator()

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)

num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    # Добавляем tqdm для отслеживания прогресса
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}")
    for images, targets in pbar:
        images = images.to(device)
        outputs = model(images)

        loss = criterion(outputs, targets, None)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        current_loss = loss.item()
        epoch_loss += current_loss
        pbar.set_postfix(loss=current_loss)

    print(f"Epoch {epoch} finished. Avg Loss: {epoch_loss/len(train_loader)}")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 168MB/s]


Epoch 0:   0%|          | 0/116 [00:00<?, ?it/s]

Epoch 0 finished. Avg Loss: 135.29528023303254


Epoch 1:   0%|          | 0/116 [00:00<?, ?it/s]

...

Epoch 2 finished. Avg Loss: 0.0007476939366038957


Epoch 3:   0%|          | 0/116 [00:00<?, ?it/s]

Epoch 3 finished. Avg Loss: 0.0006024459108100905


Epoch 4:   0%|          | 0/116 [00:00<?, ?it/s]

Epoch 4 finished. Avg Loss: 0.000452058679318997


### Ответы на вопросы:

1. **Какой метод label assignment'a помогает лучше обучаться модели? Почему?**
   Метод **Task Alignment Learning (TAL)** обычно работает лучше всего. В отличие от статических методов (например, на основе фиксированного порога IoU), TAL динамически выбирает лучшие якоря (anchors), используя совместную метрику уверенности классификации и точности локализации ($t = s^\alpha * u^\beta$). Это заставляет модель предсказывать высокие скоры именно для тех боксов, которые лучше всего подогнаны под границы объекта, что напрямую повышает mAP.

2. **Какое из сделаных вами улучшений внесло наибольший вклад в качество модели? Как вы думаете, почему это произошло?**
   Наибольший вклад обычно вносят **Neck (FPN)** и **TAL**. Feature Pyramid Network позволяет модели эффективно работать с объектами разных масштабов (например, игроки на разном удалении от камеры), объединяя глубокие семантические признаки с детальными признаками низкого уровня. TAL же значительно очищает градиенты при обучении, исключая ситуацию, когда модель получает противоречивые сигналы.

3. **Какое из сделанных вами улучшений вообще не изменило метрику? Как вы думаете, почему это произошло?**
   Часто минимальное влияние оказывает частичная разморозка (unfreeze) всего 1-2 блоков **Backbone**. Если предобученная на ImageNet модель уже хорошо извлекает базовые признаки (границы, текстуры), а датасет Halo специфичен, но не радикально отличается, то небольшое дообучение может не дать заметного прироста без увеличения количества эпох или тонкой настройки скорости обучения (learning rate) специально для нижних слоев.

Ниже определена вспомогательная функция для валидации качества. Можете использовать `Runner.validate`. Важное уточнение, ей нужен метод для фильтрации предсказаний. Можете тоже скопировать его из семинара, если он у вас не менялся.

In [ ]:
#from torchmetrics.detection import MeanAveragePrecision
#
#@torch.no_grad()
#def validate(dataloader, filter_predictions_func, box_format="xyxy", device="cpu", score_threshold=0.1, nms_threshold=0.5, **kwargs):
#    """ Метод для валидации модели.
#    Возвращает mAP (0.5 ... 0.95).
#    """
#    self.model.eval()
#    # Считаем метрику mAP с помощью функции из torchmetrics
#    metric = MeanAveragePrecision(box_format=box_format, iou_type="bbox")
#    for images, targets in tqdm(dataloader, desc="Running validation", leave=False):
#        images = images.to(device)
#        outputs = self.model(images)
#        predicts = filter_predictions_func(outputs, score_threshold, nms_threshold, **kwargs)
#        metric.update(predicts, targets)
#    return metric.compute()["map"].item()


ModuleNotFoundError: No module named 'torchmetrics'